In [104]:
#imports
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from statsmodels.stats.weightstats import ttest_ind
import scipy.stats as stats
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.stats.api as sms
import statsmodels.api as sm

## Processing DPW data

In [105]:
dpw_raw_pm25 = pd.read_csv("raw PM data/dpw pm25 raw.csv")
dpw_raw_pm25.head()

,timestamp,id,timestamp_local,sn,rh,temp,lat,lon,device_state,pm1,...,no,no2,o3,pm1_model_id,pm25_model_id,pm10_model_id,co_model_id,no_model_id,no2_model_id,o3_model_id
0,2025-12-31T23:59:55Z,577611969,2025-12-31T18:59:55Z,MOD-00811,49.8,-2.1,41.79521,-71.39783,ACTIVE,6.207,...,6.865,27.876,18.821,16235.0,16236.0,16237.0,18742.0,18743.0,18744.0,18745.0
1,2025-12-31T23:58:55Z,577611968,2025-12-31T18:58:55Z,MOD-00811,49.9,-2.0,41.79521,-71.39783,ACTIVE,6.826,...,6.180,27.884,18.801,16235.0,16236.0,16237.0,18742.0,18743.0,18744.0,18745.0
2,2025-12-31T23:57:55Z,577610031,2025-12-31T18:57:55Z,MOD-00811,49.5,-2.0,41.79521,-71.39783,ACTIVE,5.524,...,5.140,26.688,20.569,16235.0,16236.0,16237.0,18742.0,18743.0,18744.0,18745.0
3,2025-12-31T23:56:55Z,577610033,2025-12-31T18:56:55Z,MOD-00811,49.3,-2.0,41.79521,-71.39783,ACTIVE,6.057,...,4.105,25.880,21.287,16235.0,16236.0,16237.0,18742.0,18743.0,18744.0,18745.0
4,2025-12-31T23:55:55Z,577610032,2025-12-31T18:55:55Z,MOD-00811,49.3,-2.0,41.79521,-71.39783,ACTIVE,5.706,...,6.854,25.880,21.287,16235.0,16236.0,16237.0,18742.0,18743.0,18744.0,18745.0


In [106]:
#remove unnecessary columns, reformat timestamp, filter and resample
dpw_raw_pm25 = dpw_raw_pm25[["timestamp", "rh", "temp", "pm25"]].copy()
dpw_raw_pm25["datetime_utc"] = pd.to_datetime(dpw_raw_pm25["timestamp"], utc=True)
dpw_raw_pm25 = dpw_raw_pm25.drop(columns=["timestamp"])

start = pd.Timestamp("2025-01-02", tz="UTC")
end = pd.Timestamp("2025-08-03 23:59:59", tz="UTC")

dpw_raw_pm25 = dpw_raw_pm25[(dpw_raw_pm25["datetime_utc"] >= start) & (dpw_raw_pm25["datetime_utc"] <= end)]

dpw_raw_pm25 = dpw_raw_pm25.set_index("datetime_utc")
dpw_m1_pm25 = dpw_raw_pm25.resample("1h").mean()
dpw_m1_pm25 = dpw_m1_pm25.reset_index()

dpw_m1_pm25.head()

,datetime_utc,rh,temp,pm25
0,2025-01-02 00:00:00+00:00,70.305000,7.703333,0.841567
1,2025-01-02 01:00:00+00:00,71.536667,6.006667,0.478867
2,2025-01-02 02:00:00+00:00,72.690000,5.530000,0.458833
3,2025-01-02 03:00:00+00:00,69.690000,5.386667,0.442483
4,2025-01-02 04:00:00+00:00,66.933333,5.158333,0.464817


In [107]:
#now need to add in WD WS data from allens Ave
aa_raw_wind = pd.read_csv("raw wind data/AA wind raw.csv")
aa_raw_wind = aa_raw_wind[["date", "ws", "wd"]].copy()
aa_raw_wind["datetime_utc"] = pd.to_datetime(aa_raw_wind["date"], utc=True)
aa_raw_wind = aa_raw_wind.drop(columns=["date"])
aa_raw_wind.head()

,ws,wd,datetime_utc
0,0.9,80.0,2025-01-01 00:00:00+00:00
1,2.1,30.0,2025-01-01 01:00:00+00:00
2,1.5,20.0,2025-01-01 02:00:00+00:00
3,1.5,70.0,2025-01-01 03:00:00+00:00
4,1.5,20.0,2025-01-01 04:00:00+00:00


In [108]:
#merge on datetime
dpw_m2_pm25 = pd.merge(dpw_m1_pm25, aa_raw_wind, on="datetime_utc", how="left")
dpw_m2_pm25.head()

,datetime_utc,rh,temp,pm25,ws,wd
0,2025-01-02 00:00:00+00:00,70.305000,7.703333,0.841567,3.00,290.0
1,2025-01-02 01:00:00+00:00,71.536667,6.006667,0.478867,4.85,300.0
2,2025-01-02 02:00:00+00:00,72.690000,5.530000,0.458833,3.30,280.0
3,2025-01-02 03:00:00+00:00,69.690000,5.386667,0.442483,6.20,270.0
4,2025-01-02 04:00:00+00:00,66.933333,5.158333,0.464817,8.20,280.0


In [109]:
#preparation for ML

#make wind into u-v components rather than polar
dpw_m2_pm25["wd_rad"] = np.deg2rad(dpw_m2_pm25["wd"])
dpw_m2_pm25["wind_u"] = -dpw_m2_pm25["ws"] * np.sin(dpw_m2_pm25["wd_rad"])
dpw_m2_pm25["wind_v"] = -dpw_m2_pm25["ws"] * np.cos(dpw_m2_pm25["wd_rad"])
dpw_m2_pm25 = dpw_m2_pm25.drop(columns=['wd_rad'])

#hour sin/cos to capture cylical patterns
dpw_m2_pm25["hour"] = dpw_m2_pm25["datetime_utc"].dt.hour
dpw_m2_pm25["hour_sin"] = np.sin(2 * np.pi * dpw_m2_pm25["hour"] / 24)
dpw_m2_pm25["hour_cos"] = np.cos(2 * np.pi * dpw_m2_pm25["hour"] / 24)

#create lags
dpw_m2_pm25["pm_lag1"] = dpw_m2_pm25["pm25"].shift(1)
dpw_m2_pm25["pm_lag2"] = dpw_m2_pm25["pm25"].shift(2)
dpw_m2_pm25["pm_lag3"] = dpw_m2_pm25["pm25"].shift(3)
dpw_m2_pm25["pm_lag6"] = dpw_m2_pm25["pm25"].shift(6)
dpw_final_pm25 = dpw_m2_pm25.dropna()

dpw_final_pm25.head()
dpw_final_pm25.to_csv("final PM data/dpw_final_pm25.csv")

## Processing PEMA data

In [110]:
pema_raw_pm25 = pd.read_csv("raw PM data/pema pm25 raw.csv")
pema_raw_pm25.head()

,timestamp,id,timestamp_local,sn,rh,temp,lat,lon,device_state,wd,...,no2,o3,pm1_model_id,pm25_model_id,pm10_model_id,co_model_id,no_model_id,no2_model_id,o3_model_id,ws_scalar
0,2025-12-31T23:59:48Z,577612542,2025-12-31T18:59:48Z,MOD-00810,50.5,-2.3,41.8522,-71.4198,ACTIVE,246.5,...,23.652,18.138,16071.0,16072.0,16073.0,18738.0,18739.0,18740.0,18741.0,1.72
1,2025-12-31T23:58:48Z,577610523,2025-12-31T18:58:48Z,MOD-00810,50.2,-2.2,41.8522,-71.4198,ACTIVE,243.9,...,24.142,18.481,16071.0,16072.0,16073.0,18738.0,18739.0,18740.0,18741.0,3.64
2,2025-12-31T23:57:48Z,577610521,2025-12-31T18:57:48Z,MOD-00810,50.3,-2.2,41.8522,-71.4198,ACTIVE,259.8,...,24.126,19.385,16071.0,16072.0,16073.0,18738.0,18739.0,18740.0,18741.0,2.70
3,2025-12-31T23:56:48Z,577610524,2025-12-31T18:56:48Z,MOD-00810,50.0,-2.2,41.8522,-71.4198,ACTIVE,241.9,...,23.334,19.654,16071.0,16072.0,16073.0,18738.0,18739.0,18740.0,18741.0,3.30
4,2025-12-31T23:55:48Z,577610522,2025-12-31T18:55:48Z,MOD-00810,49.8,-2.2,41.8522,-71.4198,ACTIVE,261.7,...,21.268,23.022,16071.0,16072.0,16073.0,18738.0,18739.0,18740.0,18741.0,3.79


In [111]:
#remove columns, format, etc. Similar to process before but with no merging needed
pema_raw_pm25 = pema_raw_pm25[["timestamp", "rh", "temp", "wd", "ws", "pm25"]].copy()
pema_raw_pm25["datetime_utc"] = pd.to_datetime(pema_raw_pm25["timestamp"], utc=True)
pema_raw_pm25 = pema_raw_pm25.drop(columns=["timestamp"])

pema_raw_pm25 = pema_raw_pm25[(pema_raw_pm25["datetime_utc"] >= start) & (pema_raw_pm25["datetime_utc"] <= end)]

#resample
pema_raw_pm25 = pema_raw_pm25.set_index("datetime_utc")
pema_raw_pm25 = pema_raw_pm25.resample("1h").mean()

pema_m_pm25 = pema_raw_pm25.reset_index()

pema_m_pm25.head()

,datetime_utc,rh,temp,wd,ws,pm25
0,2025-01-02 00:00:00+00:00,70.066667,7.808333,313.078333,5.714000,0.096833
1,2025-01-02 01:00:00+00:00,73.005000,5.875000,296.005000,4.359833,0.050633
2,2025-01-02 02:00:00+00:00,72.535000,5.458333,279.295000,4.301333,0.047417
3,2025-01-02 03:00:00+00:00,69.446667,5.348333,283.990000,4.909000,0.033750
4,2025-01-02 04:00:00+00:00,67.353333,5.035000,285.388333,4.950167,0.049150


In [112]:
#create features for ML as before

pema_m_pm25["wd_rad"] = np.deg2rad(pema_m_pm25["wd"])
pema_m_pm25["wind_u"] = -pema_m_pm25["ws"] * np.sin(pema_m_pm25["wd_rad"])
pema_m_pm25["wind_v"] = -pema_m_pm25["ws"] * np.cos(pema_m_pm25["wd_rad"])
pema_m_pm25 = pema_m_pm25.drop(columns=["wd_rad"])

pema_m_pm25["hour"] = pema_m_pm25["datetime_utc"].dt.hour
pema_m_pm25["hour_sin"] = np.sin(2 * np.pi * pema_m_pm25["hour"] / 24)
pema_m_pm25["hour_cos"] = np.cos(2 * np.pi * pema_m_pm25["hour"] / 24)

pema_m_pm25["pm_lag1"] = pema_m_pm25["pm25"].shift(1)
pema_m_pm25["pm_lag2"] = pema_m_pm25["pm25"].shift(2)
pema_m_pm25["pm_lag3"] = pema_m_pm25["pm25"].shift(3)
pema_m_pm25["pm_lag6"] = pema_m_pm25["pm25"].shift(6)
pema_final_pm25 = pema_m_pm25.dropna()

pema_final_pm25.head()
pema_final_pm25.to_csv("final PM data/pema_final_pm25.csv")

## Processing PHA data

In [113]:
pha_raw_pm25 = pd.read_csv("raw PM data/pha pm25 raw.csv")
pha_raw_pm25.head()

,timestamp,id,timestamp_local,sn,rh,temp,lat,lon,device_state,wd,...,no2,o3,pm1_model_id,pm25_model_id,pm10_model_id,co_model_id,no_model_id,no2_model_id,o3_model_id,ws_scalar
0,2025-12-31T23:59:37Z,577610952,2025-12-31T18:59:37Z,MOD-00812,52.7,-2.9,41.8171,-71.4553,ACTIVE,265.2,...,27.697,14.423,16238.0,16239.0,16240.0,18746.0,18747.0,18748.0,18749.0,7.20
1,2025-12-31T23:58:37Z,577610943,2025-12-31T18:58:37Z,MOD-00812,52.3,-2.9,41.8171,-71.4553,ACTIVE,279.0,...,27.339,15.945,16238.0,16239.0,16240.0,18746.0,18747.0,18748.0,18749.0,9.30
2,2025-12-31T23:57:37Z,577610942,2025-12-31T18:57:37Z,MOD-00812,52.3,-2.9,41.8171,-71.4553,ACTIVE,229.1,...,27.759,15.365,16238.0,16239.0,16240.0,18746.0,18747.0,18748.0,18749.0,8.77
3,2025-12-31T23:56:37Z,577610941,2025-12-31T18:56:37Z,MOD-00812,52.6,-2.9,41.8171,-71.4553,ACTIVE,261.5,...,28.132,12.939,16238.0,16239.0,16240.0,18746.0,18747.0,18748.0,18749.0,7.28
4,2025-12-31T23:55:37Z,577608883,2025-12-31T18:55:37Z,MOD-00812,52.7,-2.9,41.8171,-71.4553,ACTIVE,264.2,...,28.955,12.685,16238.0,16239.0,16240.0,18746.0,18747.0,18748.0,18749.0,6.23


In [114]:
#remove columns, format, etc. Similar to process before but with no merging needed
pha_raw_pm25 = pha_raw_pm25[["timestamp", "rh", "temp", "wd", "ws", "pm25"]].copy()
pha_raw_pm25["datetime_utc"] = pd.to_datetime(pha_raw_pm25["timestamp"], utc=True)
pha_raw_pm25 = pha_raw_pm25.drop(columns=["timestamp"])

pha_raw_pm25 = pha_raw_pm25[(pha_raw_pm25["datetime_utc"] >= start) & (pha_raw_pm25["datetime_utc"] <= end)]

#resample
pha_raw_pm25 = pha_raw_pm25.set_index("datetime_utc")
pha_raw_pm25 = pha_raw_pm25.resample("1h").mean()

pha_m_pm25 = pha_raw_pm25.reset_index()

pha_m_pm25.head()

,datetime_utc,rh,temp,wd,ws,pm25
0,2025-01-02 00:00:00+00:00,71.023333,7.333333,273.293333,2.491500,0.389583
1,2025-01-02 01:00:00+00:00,73.566667,5.595000,273.903333,2.160667,0.214200
2,2025-01-02 02:00:00+00:00,73.408333,5.010000,271.506667,2.746000,0.192533
3,2025-01-02 03:00:00+00:00,70.981667,4.791667,269.335000,2.488000,0.198267
4,2025-01-02 04:00:00+00:00,68.388333,4.468333,274.261667,2.607333,0.168467


In [115]:
#create features for ML as before

pha_m_pm25["wd_rad"] = np.deg2rad(pha_m_pm25["wd"])
pha_m_pm25["wind_u"] = -pha_m_pm25["ws"] * np.sin(pha_m_pm25["wd_rad"])
pha_m_pm25["wind_v"] = -pha_m_pm25["ws"] * np.cos(pha_m_pm25["wd_rad"])
pha_m_pm25 = pha_m_pm25.drop(columns=["wd_rad"])

pha_m_pm25["hour"] = pha_m_pm25["datetime_utc"].dt.hour
pha_m_pm25["hour_sin"] = np.sin(2 * np.pi * pha_m_pm25["hour"] / 24)
pha_m_pm25["hour_cos"] = np.cos(2 * np.pi * pha_m_pm25["hour"] / 24)

pha_m_pm25["pm_lag1"] = pha_m_pm25["pm25"].shift(1)
pha_m_pm25["pm_lag2"] = pha_m_pm25["pm25"].shift(2)
pha_m_pm25["pm_lag3"] = pha_m_pm25["pm25"].shift(3)
pha_m_pm25["pm_lag6"] = pha_m_pm25["pm25"].shift(6)
pha_final_pm25 = pha_m_pm25.dropna()

pha_final_pm25.head()
pha_final_pm25.to_csv("final PM data/ph_final_pm25.csv")